In [0]:
# Original SP

In [0]:
%sql
/*
For a given user, are they a reporter
*/
CREATE OR REPLACE PROCEDURE gold.sp_IsReporter

(
    IN par_userId INT
)
LANGUAGE SQL
SQL SECURITY INVOKER 
AS 
BEGIN

  SELECT DISTINCT
  CASE WHEN 
    ( 
      CASE WHEN UAL.userId IS NULL THEN 0 ELSE 1 END   +
      CASE WHEN UGR.userId IS NULL THEN 0 ELSE 1 END   +
      CASE WHEN URU.reportingUserId IS NULL THEN 0 ELSE 1 END
    ) 
    = 0 THEN 0 ELSE 1 END AS IsReporter

  FROM bronze_learning_hub.hub_user AU
  -- Location based permission
  LEFT JOIN silver.useradminlocation_v UAL 
      ON AU.Id = UAL.userId
      AND NOT UAL.Deleted 
      AND UAL.adminLocationId NOT IN (1,190822) -- Unknown or Unselected
  -- User group based permission
  LEFT JOIN bronze_learning_hub.elfh_usergroupreportertbl UGR 
      ON UGR.userId = AU.Id 
      AND NOT UGR.deleted
  -- User based permission
  LEFT JOIN bronze_learning_hub.elfh_userreportingusertbl URU
    ON URU.reportingUserId = AU.ID
      AND NOT URU.Deleted

  WHERE AU.Id = par_userId --:Reporter
  AND NOT AU.Deleted

  ;
END;

